# โครงงานวิเคราะห์ข้อมูล Advertising (ฉบับอธิบายทุกบรรทัด)
โค้ดชุดเดียวกับ `Advertising_analysis.ipynb` แต่ **ทุกบรรทัดมีคอมเมนต์ภาษาไทย** (ข้อความหลังเครื่องหมาย `#`) บอกว่าบรรทัดนั้นทำอะไร
คอมเมนต์ไม่มีผลต่อการทำงานของโปรแกรม ใช้อ่านเพื่อทำความเข้าใจและเตรียมตอบอาจารย์

**วิธีใช้:** วาง `Advertising.csv` ไว้โฟลเดอร์เดียวกับไฟล์นี้ แล้วกด **Restart → Run All**

**ศัพท์ที่เจอบ่อย:** *DataFrame* = ตารางข้อมูลของ pandas · *คอลัมน์* = ตัวแปรหนึ่งตัว · *loop* = วนทำซ้ำ · *train* = ข้อมูลสอนโมเดล · *test* = ข้อมูลสอบโมเดล

## 1. นำเข้า library และอ่านข้อมูล

In [ ]:
import os  # เรียกโมดูล os ไว้จัดการโฟลเดอร์/ไฟล์ในเครื่อง
import numpy as np  # เรียก NumPy ตั้งชื่อย่อว่า np ไว้คำนวณตัวเลข
import pandas as pd  # เรียก pandas ตั้งชื่อย่อว่า pd ไว้จัดการข้อมูลแบบตาราง
import matplotlib.pyplot as plt  # เรียก matplotlib ตั้งชื่อย่อว่า plt ไว้วาดกราฟ

os.makedirs('output', exist_ok=True)  # สร้างโฟลเดอร์ชื่อ output (exist_ok=True คือถ้ามีอยู่แล้วก็ไม่ต้อง error)

df = pd.read_csv('Advertising.csv')  # อ่านไฟล์ CSV เก็บไว้ในตัวแปร df (df = DataFrame คือตาราง)
df = df.drop(columns='Unnamed: 0')  # ลบคอลัมน์เลขลำดับที่ไม่ใช่ข้อมูลจริง แล้วเก็บตารางใหม่ทับ df
df.head()  # แสดง 5 แถวแรกของตาราง เพื่อดูว่าอ่านข้อมูลมาถูกไหม

## 2. ตั้งค่าฟอนต์ภาษาไทยให้กราฟ

In [ ]:
import logging  # เรียกโมดูล logging ที่ควบคุมข้อความเตือนของโปรแกรม
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)  # แสดงเฉพาะ error ร้ายแรง จึงไม่มีคำเตือน "font not found" ขึ้นรก

plt.rcParams['font.family'] = ['TH Sarabun New', 'Tahoma']  # ตั้งฟอนต์ของกราฟ ลองตัวแรกก่อน ถ้าไม่มีในเครื่องใช้ตัวถัดไป
plt.rcParams['font.size'] = 16  # ตั้งขนาดตัวอักษรในกราฟเป็น 16 (TH Sarabun New ตัวเล็ก จึงตั้งให้ใหญ่ขึ้น)
plt.rcParams['axes.unicode_minus'] = False  # ให้เครื่องหมายลบแสดงเป็นขีดธรรมดา กันแสดงเป็นสี่เหลี่ยมเมื่อใช้ฟอนต์ไทย

## 3. ตรวจสอบข้อมูลเบื้องต้น

In [ ]:
print('จำนวนแถว, คอลัมน์ =', df.shape)  # df.shape คือ (จำนวนแถว, จำนวนคอลัมน์) พิมพ์พร้อมข้อความนำ
print()  # พิมพ์บรรทัดว่างให้อ่านง่าย
df.info()  # สรุปชนิดข้อมูล และจำนวนค่าที่ไม่ว่างของแต่ละคอลัมน์

In [ ]:
print('ค่าสูญหายในแต่ละคอลัมน์')  # พิมพ์หัวข้อ
print(df.isnull().sum())  # isnull() ตรวจว่าช่องไหนว่าง (True/False) แล้ว sum() นับจำนวน True ต่อคอลัมน์
print()  # บรรทัดว่าง
print('จำนวนแถวซ้ำ =', df.duplicated().sum())  # duplicated() ตรวจแถวที่ซ้ำกับแถวก่อนหน้า แล้ว sum() นับจำนวน

## 4. สถิติเชิงพรรณนา

In [ ]:
summary = pd.DataFrame({  # สร้างตารางใหม่ชื่อ summary จากพจนานุกรม (ชื่อคอลัมน์: ค่า)
    'ค่าเฉลี่ย': df.mean(),  # ค่าเฉลี่ยของแต่ละตัวแปร
    'ส.บ.ม. (ddof=1)': df.std(ddof=1),  # ส่วนเบี่ยงเบนมาตรฐานแบบตัวอย่าง (ddof=1 คือหารด้วย n-1)
    'ต่ำสุด': df.min(),  # ค่าน้อยที่สุด
    'มัธยฐาน': df.median(),  # ค่ากลางเมื่อเรียงข้อมูล
    'สูงสุด': df.max()  # ค่ามากที่สุด
}).round(2)  # ปัดทุกค่าเป็นทศนิยม 2 ตำแหน่ง
summary.to_csv('output/ตารางสถิติ.csv', encoding='utf-8-sig')  # บันทึกเป็นไฟล์ CSV (utf-8-sig ทำให้ Excel อ่านภาษาไทยออก)
summary  # แสดงตารางในโน้ตบุ๊ก

## 5. ตรวจหาค่าผิดปกติด้วยกฎ 1.5 × IQR

In [ ]:
rows = []  # สร้างลิสต์ว่างไว้เก็บผลของแต่ละตัวแปร
for col in df.columns:  # วนทีละคอลัมน์ (col จะเป็น TV, Radio, Newspaper, Sales ตามลำดับ)
    q1 = df[col].quantile(0.25)  # Q1 ค่าที่ตำแหน่ง 25% ของข้อมูล
    q3 = df[col].quantile(0.75)  # Q3 ค่าที่ตำแหน่ง 75% ของข้อมูล
    iqr = q3 - q1  # IQR คือช่วงของข้อมูลตรงกลาง 50%
    low = q1 - 1.5 * iqr  # ขอบล่าง ต่ำกว่านี้ถือว่าผิดปกติ
    high = q3 + 1.5 * iqr  # ขอบบน สูงกว่านี้ถือว่าผิดปกติ
    n = ((df[col] < low) | (df[col] > high)).sum()  # นับค่าที่ต่ำกว่าขอบล่าง หรือ (|) สูงกว่าขอบบน
    rows.append([col, q1, q3, iqr, low, high, n])  # เก็บผลของตัวแปรนี้ต่อท้ายลิสต์

outlier_table = pd.DataFrame(rows, columns=['ตัวแปร', 'Q1', 'Q3', 'IQR', 'ขอบล่าง', 'ขอบบน', 'จำนวนค่าผิดปกติ']).round(2)  # แปลงลิสต์เป็นตาราง ตั้งชื่อคอลัมน์ ปัดทศนิยม 2 ตำแหน่ง
outlier_table.to_csv('output/ตารางค่าผิดปกติ.csv', index=False, encoding='utf-8-sig')  # บันทึกเป็น CSV (index=False คือไม่เอาเลขแถวไปด้วย)
outlier_table  # แสดงตาราง

In [ ]:
q1 = df['Newspaper'].quantile(0.25)  # หา Q1 ของ Newspaper
q3 = df['Newspaper'].quantile(0.75)  # หา Q3 ของ Newspaper
iqr = q3 - q1  # คำนวณ IQR
df[(df['Newspaper'] < q1 - 1.5*iqr) | (df['Newspaper'] > q3 + 1.5*iqr)]  # กรองเฉพาะแถวที่ Newspaper ผิดปกติ มาแสดง

## 6. กราฟสำรวจข้อมูล

### 6.1 ฮิสโตแกรม

In [ ]:
names = {'TV': 'โฆษณาทางทีวี (TV)', 'Radio': 'โฆษณาทางวิทยุ (Radio)',  # พจนานุกรมจับคู่ชื่อคอลัมน์กับชื่อภาษาไทย ไว้ใช้เป็นชื่อกราฟ
         'Newspaper': 'โฆษณาทางหนังสือพิมพ์ (Newspaper)', 'Sales': 'ยอดขาย (Sales)'}  # (ต่อจากบรรทัดบน)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))  # สร้างภาพใหญ่ 1 ภาพ แบ่งเป็นกราฟย่อย 2 แถว 2 คอลัมน์ ขนาด 12x9 นิ้ว
for ax, col in zip(axes.flatten(), df.columns):  # flatten() ทำให้กราฟย่อย 4 อันเรียงเป็นแถวเดียว zip() จับคู่กราฟย่อยกับชื่อคอลัมน์ทีละคู่
    ax.hist(df[col], bins=15, color='steelblue', edgecolor='white')  # วาดฮิสโตแกรม 15 แท่ง สีน้ำเงิน ขอบสีขาว
    ax.set_title('ฮิสโตแกรมของ ' + names[col])  # ตั้งชื่อกราฟย่อย โดยดึงชื่อไทยจาก names
    ax.set_xlabel(col)  # ตั้งชื่อแกน x เป็นชื่อตัวแปร
    ax.set_ylabel('ความถี่')  # ตั้งชื่อแกน y
plt.tight_layout()  # จัดระยะห่างไม่ให้ชื่อกราฟซ้อนกัน
plt.savefig('output/histogram.png', dpi=200)  # บันทึกรูปเป็นไฟล์ PNG ความละเอียด 200 dpi
plt.show()  # แสดงกราฟในโน้ตบุ๊ก

### 6.2 แผนภาพกล่อง

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 5))  # สร้างภาพที่มีกราฟย่อย 1 แถว 4 อัน (หนึ่งอันต่อหนึ่งตัวแปร)
for ax, col in zip(axes, df.columns):  # จับคู่กราฟย่อยกับชื่อคอลัมน์ทีละคู่
    ax.boxplot(df[col])  # วาดแผนภาพกล่องของคอลัมน์นั้น จุดนอกหนวดคือค่าผิดปกติ
    ax.set_title(col)  # ตั้งชื่อกราฟย่อยเป็นชื่อตัวแปร
    ax.set_xticks([])  # ซ่อนตัวเลขบนแกน x ที่ไม่มีความหมาย
fig.suptitle('แผนภาพกล่องของแต่ละตัวแปร')  # ตั้งชื่อรวมที่ด้านบนของทั้งภาพ
plt.tight_layout()  # จัดระยะห่างอัตโนมัติ
plt.savefig('output/boxplot.png', dpi=200)  # บันทึกเป็นไฟล์ภาพ
plt.show()  # แสดงกราฟ

### 6.3 กราฟการกระจาย (scatter)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))  # สร้างกราฟย่อย 1 แถว 3 อัน
for ax, col in zip(axes, ['TV', 'Radio', 'Newspaper']):  # วนทีละช่องทางโฆษณา จับคู่กับกราฟย่อย
    ax.scatter(df[col], df['Sales'], color='steelblue', alpha=0.7)  # วาดจุด x=งบโฆษณา y=ยอดขาย alpha=0.7 ทำให้จุดโปร่งแสงเล็กน้อย
    r = df[col].corr(df['Sales'])  # คำนวณค่าสหสัมพันธ์ r ระหว่างช่องทางนี้กับ Sales
    ax.set_title(col + ' กับ Sales (r = ' + str(round(r, 3)) + ')')  # ชื่อกราฟ พร้อมค่า r ปัดเป็นทศนิยม 3 ตำแหน่ง (str() แปลงตัวเลขเป็นข้อความ)
    ax.set_xlabel('งบโฆษณา ' + col)  # ชื่อแกน x
    ax.set_ylabel('Sales')  # ชื่อแกน y
plt.tight_layout()  # จัดระยะห่างอัตโนมัติ
plt.savefig('output/scatter.png', dpi=200)  # บันทึกเป็นไฟล์ภาพ
plt.show()  # แสดงกราฟ

## 7. ค่าสหสัมพันธ์

In [ ]:
corr = df.corr().round(3)  # คำนวณตารางสหสัมพันธ์ของทุกคู่ตัวแปร ปัดทศนิยม 3 ตำแหน่ง
corr.to_csv('output/ตารางสหสัมพันธ์.csv', encoding='utf-8-sig')  # บันทึกเป็นไฟล์ CSV
corr  # แสดงตาราง

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))  # สร้างภาพ 1 กราฟ ขนาด 6.5x5.5 นิ้ว
im = ax.imshow(corr.values, cmap='Blues', vmin=0, vmax=1)  # วาดตารางตัวเลขเป็นสี (ยิ่งเข้ม ยิ่งสัมพันธ์มาก) กำหนดสเกลสี 0 ถึง 1
ax.set_xticks(range(len(corr.columns)))  # กำหนดตำแหน่งป้ายบนแกน x ตามจำนวนคอลัมน์
ax.set_xticklabels(corr.columns)  # ใส่ชื่อตัวแปรเป็นป้ายแกน x
ax.set_yticks(range(len(corr.columns)))  # กำหนดตำแหน่งป้ายบนแกน y
ax.set_yticklabels(corr.columns)  # ใส่ชื่อตัวแปรเป็นป้ายแกน y
for i in range(len(corr)):  # วนทีละแถวของตาราง
    for j in range(len(corr)):  # ในแต่ละแถว วนทีละคอลัมน์ (loop ซ้อน loop)
        v = corr.values[i, j]  # ดึงค่าสหสัมพันธ์ที่ตำแหน่งแถว i คอลัมน์ j
        ax.text(j, i, str(v), ha='center', va='center', color='white' if v > 0.6 else 'black')  # เขียนตัวเลขกลางช่อง ถ้าค่า > 0.6 ใช้ตัวอักษรขาว (พื้นเข้ม) ไม่งั้นใช้ดำ
fig.colorbar(im)  # แสดงแถบสีอธิบายความหมายของสี
ax.set_title('สหสัมพันธ์ระหว่างตัวแปร')  # ตั้งชื่อกราฟ
plt.tight_layout()  # จัดระยะห่างอัตโนมัติ
plt.savefig('output/heatmap.png', dpi=200)  # บันทึกเป็นไฟล์ภาพ
plt.show()  # แสดงกราฟ

In [ ]:
print(corr['Sales'].drop('Sales').sort_values(ascending=False))  # ดึงคอลัมน์ Sales ตัดแถว Sales เองทิ้ง (สหสัมพันธ์กับตัวเอง = 1) แล้วเรียงจากมากไปน้อย

## 8. แบ่งข้อมูล Train / Test

In [ ]:
from sklearn.model_selection import train_test_split  # เรียกฟังก์ชันแบ่งข้อมูลสอน/ข้อมูลสอบ
from sklearn.pipeline import make_pipeline  # เรียกตัวต่อขั้นตอนหลายขั้นให้เป็นสายเดียว
from sklearn.preprocessing import PolynomialFeatures, StandardScaler  # PolynomialFeatures เพิ่มพจน์ยกกำลัง, StandardScaler ปรับสเกลตัวแปร
from sklearn.linear_model import LinearRegression  # โมเดลถดถอยเชิงเส้น
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error  # ฟังก์ชันวัดผล R², MSE, MAE

X = df[['TV', 'Radio', 'Newspaper']]  # X คือตัวแปรที่ใช้ทำนาย (ตารางที่มี 3 คอลัมน์ จึงใช้ [[ ]] สองชั้น)
y = df['Sales']  # y คือตัวแปรที่อยากทำนาย (คอลัมน์เดียว)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)  # สุ่มแบ่ง 70% เป็น train (_tr) และ 30% เป็น test (_te) random_state=0 ล็อกให้แบ่งเหมือนเดิมทุกครั้ง
print('train =', X_tr.shape[0], 'แถว, test =', X_te.shape[0], 'แถว')  # shape[0] คือจำนวนแถว พิมพ์ขนาดของ train และ test

## 9. Linear Regression

In [ ]:
lin = LinearRegression()  # สร้างโมเดลเส้นตรง (ยังไม่ได้สอน)
lin.fit(X_tr, y_tr)  # สอนโมเดลด้วยข้อมูล train เท่านั้น (fit = หาสมการที่เหมาะสมที่สุด)

print('จุดตัด (intercept) =', round(lin.intercept_, 4))  # intercept_ คือค่าคงที่ในสมการ (ยอดขายเมื่องบทุกช่องเป็น 0)
for name, c in zip(X.columns, lin.coef_):  # จับคู่ชื่อตัวแปรกับสัมประสิทธิ์ (coef_) ทีละคู่
    print('สัมประสิทธิ์ของ', name, '=', round(c, 4))  # พิมพ์ว่างบเพิ่ม 1 หน่วย ยอดขายเพิ่มเท่าไร

In [ ]:
pred_tr = lin.predict(X_tr)  # ให้โมเดลทำนาย Sales ของชุด train
pred_te = lin.predict(X_te)  # ให้โมเดลทำนาย Sales ของชุด test (ข้อมูลที่โมเดลไม่เคยเห็น)

result_lin = pd.DataFrame({  # สร้างตารางสรุปผลการวัด
    'ชุดข้อมูล': ['train', 'test'],  # คอลัมน์ชื่อชุดข้อมูล
    'R2': [r2_score(y_tr, pred_tr), r2_score(y_te, pred_te)],  # R² (ค่าจริงใส่ก่อน ค่าทำนายตามหลัง) ยิ่งใกล้ 1 ยิ่งดี
    'RMSE': [np.sqrt(mean_squared_error(y_tr, pred_tr)), np.sqrt(mean_squared_error(y_te, pred_te))],  # RMSE = รากที่สองของ MSE ความคลาดเคลื่อนหน่วยเดียวกับ Sales
    'MAE': [mean_absolute_error(y_tr, pred_tr), mean_absolute_error(y_te, pred_te)]  # MAE = ความคลาดเคลื่อนสัมบูรณ์เฉลี่ย
}).round(4)  # ปัดทศนิยม 4 ตำแหน่ง
result_lin.to_csv('output/ผลโมเดลLinear.csv', index=False, encoding='utf-8-sig')  # บันทึกเป็นไฟล์ CSV
result_lin  # แสดงตาราง

In [ ]:
plt.figure(figsize=(7, 6))  # สร้างภาพใหม่ขนาด 7x6 นิ้ว
plt.scatter(y_te, pred_te, color='steelblue', alpha=0.8)  # วาดจุด x=ยอดขายจริง y=ยอดขายที่โมเดลทำนาย (ชุด test)
lo, hi = y.min(), y.max()  # หาค่าน้อยสุดและมากสุดของ Sales ไว้กำหนดความยาวเส้นอ้างอิง
plt.plot([lo, hi], [lo, hi], color='red', linestyle='--')  # วาดเส้นประสีแดง y=x คือเส้นที่ทำนายตรงค่าจริงพอดี
plt.title('ค่าจริงเทียบกับค่าทำนาย (Linear, ชุด test)')  # ชื่อกราฟ
plt.xlabel('Sales จริง')  # ชื่อแกน x
plt.ylabel('Sales ที่โมเดลทำนาย')  # ชื่อแกน y
plt.tight_layout()  # จัดระยะห่างอัตโนมัติ
plt.savefig('output/actual_vs_predicted.png', dpi=200)  # บันทึกเป็นไฟล์ภาพ
plt.show()  # แสดงกราฟ

## 10. Polynomial Regression: เทียบหลาย degree

In [ ]:
rows = []  # ลิสต์ว่างไว้เก็บผลของแต่ละ degree
for d in [1, 2, 3, 4, 5]:  # วนลองทีละ degree (d คือระดับของพหุนาม)
    p = make_pipeline(PolynomialFeatures(d, include_bias=False),  # ขั้นที่ 1 สร้างพจน์ยกกำลังและพจน์คูณกันถึง degree d (include_bias=False คือไม่เพิ่มคอลัมน์ค่าคงที่)
                      StandardScaler(),  # ขั้นที่ 2 ปรับสเกลแต่ละคอลัมน์ให้ค่าเฉลี่ย 0 กันตัวเลขใหญ่เกินเมื่อ degree สูง
                      LinearRegression())  # ขั้นที่ 3 โมเดลเส้นตรงบนพจน์ที่สร้างขึ้น
    p.fit(X_tr, y_tr)  # สอนทั้งสายด้วยข้อมูล train เท่านั้น
    rows.append([d, p.score(X_tr, y_tr), p.score(X_te, y_te)])  # score() ให้ค่า R² เก็บ degree, R² train, R² test ต่อท้ายลิสต์

deg_table = pd.DataFrame(rows, columns=['degree', 'R2_train', 'R2_test'])  # แปลงลิสต์เป็นตาราง
deg_table['ช่องว่าง(train-test)'] = deg_table['R2_train'] - deg_table['R2_test']  # เพิ่มคอลัมน์ผลต่าง ยิ่งห่างมากยิ่งสงสัย overfitting
deg_table = deg_table.round(3)  # ปัดทศนิยม 3 ตำแหน่ง
deg_table.to_csv('output/ผลเทียบdegree.csv', index=False, encoding='utf-8-sig')  # บันทึกเป็นไฟล์ CSV
deg_table  # แสดงตาราง

In [ ]:
plt.figure(figsize=(8, 5))  # สร้างภาพใหม่ขนาด 8x5 นิ้ว
plt.plot(deg_table['degree'], deg_table['R2_train'], marker='o', label='train R²')  # เส้น R² ของ train มีจุดวงกลมที่แต่ละ degree
plt.plot(deg_table['degree'], deg_table['R2_test'], marker='o', label='test R²')  # เส้น R² ของ test
plt.xticks(deg_table['degree'])  # ให้แกน x แสดงเฉพาะเลข degree 1 ถึง 5
plt.title('R² ของ train และ test ตาม degree')  # ชื่อกราฟ
plt.xlabel('degree')  # ชื่อแกน x
plt.ylabel('R²')  # ชื่อแกน y
plt.legend()  # แสดงกล่องอธิบายว่าเส้นไหนคือ train เส้นไหนคือ test
plt.tight_layout()  # จัดระยะห่างอัตโนมัติ
plt.savefig('output/degree_compare.png', dpi=200)  # บันทึกเป็นไฟล์ภาพ
plt.show()  # แสดงกราฟ

## 11. สรุปโมเดลที่เลือก

In [ ]:
best_degree = 2  # degree ที่เลือก (แก้เลขนี้ตามผลในตารางด้านบน)

best = make_pipeline(PolynomialFeatures(best_degree, include_bias=False),  # สร้างสายโมเดลเหมือนเดิม แต่ใช้ degree ที่เลือก
                     StandardScaler(),  # ปรับสเกล
                     LinearRegression())  # โมเดลเส้นตรงบนพจน์ที่สร้าง
best.fit(X_tr, y_tr)  # สอนด้วยข้อมูล train
pb_tr = best.predict(X_tr)  # ทำนายชุด train
pb_te = best.predict(X_te)  # ทำนายชุด test

final = pd.DataFrame({  # ตารางเทียบ Linear กับโมเดลที่เลือก (ดูเฉพาะชุด test)
    'โมเดล': ['Linear (degree 1)', 'Polynomial degree ' + str(best_degree)],  # ชื่อโมเดล str() แปลงเลขเป็นข้อความเพื่อต่อสตริง
    'R2 test': [r2_score(y_te, pred_te), r2_score(y_te, pb_te)],  # R² ของแต่ละโมเดลบน test
    'RMSE test': [np.sqrt(mean_squared_error(y_te, pred_te)), np.sqrt(mean_squared_error(y_te, pb_te))],  # RMSE ของแต่ละโมเดลบน test
    'MAE test': [mean_absolute_error(y_te, pred_te), mean_absolute_error(y_te, pb_te)]  # MAE ของแต่ละโมเดลบน test
}).round(4)  # ปัดทศนิยม 4 ตำแหน่ง
final.to_csv('output/ผลเทียบโมเดล.csv', index=False, encoding='utf-8-sig')  # บันทึกเป็นไฟล์ CSV
final  # แสดงตาราง

## 12. สรุปและตีความ (เขียนเองจากผลที่ได้ด้านบน)
1. สถิติ: ตัวแปรใดเฉลี่ยสูงสุด/กระจายมากที่สุด
2. รูปร่างการกระจายของแต่ละตัวแปร
3. ค่าผิดปกติ: ตัวแปรใด กี่ค่า เก็บไว้หรือตัดทิ้ง เพราะอะไร
4. ความสัมพันธ์: ช่องทางใดสัมพันธ์กับยอดขายมากที่สุด/น้อยที่สุด
5. โมเดล: R² ของ test เท่าไร train กับ test ใกล้กันไหม
6. Polynomial: degree ไหนดีที่สุดบนชุด test และ degree สูงๆ overfitting หรือไม่
7. ข้อจำกัด: สหสัมพันธ์ไม่ใช่เหตุและผล, ข้อมูล 200 แถว, ผลมาจากการแบ่ง train/test ชุดเดียว

*(พิมพ์สรุปของพวกคุณที่นี่)*